# Module 08 — Memory Tool

Give **`claude-opus-4-8`** **memory** so it can **persist and recall facts across conversations** by reading and writing a memory store **you** control.

Memory is **client-side**: the model issues memory commands and **you execute them** against your own storage — the model never stores anything itself. It's a **BETA tool**. This builds on **Modules 00–07** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — A local memory store

The memory tool is **client-side**: the model emits commands over a **`/memories`** directory and **you implement the storage**. The commands mirror the `text_editor` tool — **`view`, `create`, `str_replace`, `insert`, `delete`, `rename`** — operating on paths under `/memories`.

For production, the SDK provides **`BetaAbstractMemoryTool`** to subclass. Here we **hand-roll a transparent local-filesystem store** so you can see exactly what happens. It maps `/memories` to a local `./memory_store` directory and refuses paths that try to escape that root.

In [ ]:
import os

class MemoryStore:
    """A transparent local-filesystem memory store rooted at `root`.

    The model addresses paths under '/memories'; we map those onto `root`.
    All access is confined to `root` for safety.
    """

    def __init__(self, root: str = "./memory_store"):
        self.root = os.path.abspath(root)
        os.makedirs(self.root, exist_ok=True)

    def _resolve(self, path: str) -> str:
        """Map a '/memories/...' path to a real path inside `root`; block escapes."""
        path = path or "/memories"
        rel = path[len("/memories"):] if path.startswith("/memories") else path
        rel = rel.lstrip("/")
        full = os.path.abspath(os.path.join(self.root, rel))
        if full != self.root and not full.startswith(self.root + os.sep):
            raise ValueError(f"path escapes memory root: {path}")
        return full

    def handle(self, tool_input: dict) -> str:
        """Dispatch a memory command; always return a string (errors included)."""
        try:
            cmd = (tool_input or {}).get("command")
            if cmd == "view":
                full = self._resolve(tool_input.get("path", "/memories"))
                if os.path.isdir(full):
                    entries = sorted(os.listdir(full))
                    return "\n".join(entries) if entries else "(empty directory)"
                with open(full, "r", encoding="utf-8") as f:
                    lines = f.read().splitlines()
                vr = tool_input.get("view_range")
                if vr and len(vr) == 2:
                    lines = lines[max(vr[0] - 1, 0):vr[1]]
                return "\n".join(lines)
            if cmd == "create":
                full = self._resolve(tool_input.get("path"))
                os.makedirs(os.path.dirname(full), exist_ok=True)
                with open(full, "w", encoding="utf-8") as f:
                    f.write(tool_input.get("file_text", ""))
                return f"created {tool_input.get('path')}"
            if cmd == "str_replace":
                full = self._resolve(tool_input.get("path"))
                with open(full, "r", encoding="utf-8") as f:
                    text = f.read()
                text = text.replace(tool_input.get("old_str", ""), tool_input.get("new_str", ""))
                with open(full, "w", encoding="utf-8") as f:
                    f.write(text)
                return f"updated {tool_input.get('path')}"
            if cmd == "insert":
                full = self._resolve(tool_input.get("path"))
                with open(full, "r", encoding="utf-8") as f:
                    lines = f.read().splitlines(keepends=True)
                idx = int(tool_input.get("insert_line", len(lines)))
                new_text = tool_input.get("insert_text", tool_input.get("text", ""))
                if not new_text.endswith("\n"):
                    new_text += "\n"
                lines.insert(idx, new_text)
                with open(full, "w", encoding="utf-8") as f:
                    f.writelines(lines)
                return f"inserted into {tool_input.get('path')}"
            if cmd == "delete":
                full = self._resolve(tool_input.get("path"))
                if os.path.isdir(full):
                    import shutil
                    shutil.rmtree(full)
                elif os.path.exists(full):
                    os.remove(full)
                return f"deleted {tool_input.get('path')}"
            if cmd == "rename":
                old = self._resolve(tool_input.get("old_path", tool_input.get("path")))
                new = self._resolve(tool_input.get("new_path"))
                os.makedirs(os.path.dirname(new), exist_ok=True)
                os.rename(old, new)
                return f"renamed to {tool_input.get('new_path')}"
            return f"error: unknown command '{cmd}'"
        except Exception as e:
            return f"error: {type(e).__name__}: {e}"

store = MemoryStore("./memory_store")
print(f"✅ MemoryStore ready at {store.root}")

## Part D — Turn 1: store a memory

The loop mirrors function calling: the model issues a **memory `tool_use`** → you run it via the store → you return a **`tool_result`** → repeat until `stop_reason` is no longer `"tool_use"`. The `run()` helper below encapsulates that loop.

In [ ]:
MEMORY_TOOL = {"type": "memory_20250818", "name": "memory"}
BETA = "context-management-2025-06-27"

def run(messages):
    """Drive the memory tool-use loop until the model produces a final answer."""
    while True:
        resp = client.beta.messages.create(
            model=MODEL,
            max_tokens=2048,
            tools=[MEMORY_TOOL],
            betas=[BETA],
            messages=messages,
        )
        if resp.stop_reason != "tool_use":
            return resp
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for block in resp.content:
            if getattr(block, "type", None) == "tool_use" and block.name == "memory":
                out = store.handle(block.input)
                results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(out),
                })
        messages.append({"role": "user", "content": results})


def final_text(resp) -> str:
    return "".join(b.text for b in resp.content if getattr(b, "type", None) == "text")

# Turn 1 — the user shares a fact to remember.
conversation_1 = [{
    "role": "user",
    "content": (
        "Please remember this for future sessions: on this project we always deploy "
        "via Cloud Build, and our preferred region is us-east5. Save it to your memory."
    ),
}]
resp1 = run(conversation_1)
print(final_text(resp1))

## Part E — Turn 2: recall in a NEW conversation

The payoff: a **brand-new conversation** — a fresh `messages` list with **no prior history** — but the **same memory store**. Claude reads its memory and recalls the fact from Turn 1.

In [ ]:
# A FRESH conversation — no history carried over, only the shared memory store.
conversation_2 = [{
    "role": "user",
    "content": "How do we deploy on this project, and what region do we prefer? Check your memory.",
}]
resp2 = run(conversation_2)
print(final_text(resp2))

# Show what actually persisted on disk.
print("\n===== ./memory_store contents =====")
for dirpath, _dirs, files in os.walk(store.root):
    for name in sorted(files):
        path = os.path.join(dirpath, name)
        rel = os.path.relpath(path, store.root)
        print(f"\n--- {rel} ---")
        with open(path, "r", encoding="utf-8") as f:
            print(f.read())

## Part F — Notes & recap

### Notes

- **Client-side:** you own the storage — a local directory here, but a database or object store in production. The model only issues commands.
- **Six commands:** `view`, `create`, `str_replace`, `insert`, `delete`, `rename` (mirroring the `text_editor` tool), all under `/memories`.
- **Pairs with context editing:** combine with context management (`clear_tool_uses_20250919`) so long sessions can offload detail to memory and clear stale tool calls from the context window.
- **Production:** subclass **`BetaAbstractMemoryTool`** instead of hand-rolling, and validate/contain all paths.
- **Beta header required** (`context-management-2025-06-27`); confirm availability for your model/region. Docs: https://docs.claude.com/en/docs/agents-and-tools/tool-use/memory-tool

### Recap

- Memory lets Claude **persist and recall facts across conversations** via a store **you** control.
- It's **client-side** and **beta**: define `{"type": "memory_20250818", "name": "memory"}`, call `client.beta.messages.create(..., betas=["context-management-2025-06-27"])`, and run the same **tool-use loop** as function calling.
- We hand-rolled a transparent `/memories` → `./memory_store` filesystem store with the six commands and path-escape protection.
- Turn 1 wrote a fact; a **fresh** Turn 2 recalled it from the shared store.

**Next: Module 09 — Prompt caching.**